# Week 21: High-Performance ML Inference, Compilers & Serving
## Notebook 03: Post-Training Quantization: AWQ, GPTQ & SmoothQuant from Scratch

### 1. Conceptual Intuition & Real-World Use Case
Serving a 70B parameter model in FP16 requires ~140 GB of GPU VRAM (two 80GB A100s).
- **Post-Training Quantization (PTQ)** compresses weights to 4 bits (INT4), reducing memory to ~35 GB (fitting on a single consumer GPU).
- However, naive Round-To-Nearest (RTN) quantization collapses perplexity.
- **Why? Activation Outliers!** A tiny fraction (0.1%) of hidden activation channels have magnitudes 100x larger than others, dominating model performance.

**Modern PTQ Breakthroughs:**
1. **AWQ (Activation-Aware Weight Quantization):** Protects the top 1% of salient weights corresponding to large activation channels.
2. **SmoothQuant:** Mathematically migrates quantization difficulty from activations to weights using an equivalent scaling transformation.

In [ ]:
import numpy as np

np.random.seed(42)

# Step 1: Simulate Real-World Activations with Channel Outliers
# Batch = 4, Dimension = 6
# Channel 2 is a massive outlier (magnitude 100x larger!)
X = np.random.randn(4, 6)
X[:, 2] *= 80.0  # Outlier activation channel

# Weight Matrix W: (6, 6)
W = np.random.randn(6, 6) * 0.5
Y_true = X @ W

print("Activation Channel Max Magnitudes across tokens:")
print(np.round(np.max(np.abs(X), axis=0), 1))

# Step 2: Naive INT4 Quantization (Round-To-Nearest - RTN)
def quantize_int4(W):
    # Scale to [-8, 7]
    scale = (np.max(W) - np.min(W)) / 15.0
    zero_point = np.round(-np.min(W) / scale) - 8.0
    W_int4 = np.clip(np.round(W / scale) + zero_point, -8, 7)
    # Dequantize back to float
    W_dequant = (W_int4 - zero_point) * scale
    return W_dequant, scale

W_rtn, _ = quantize_int4(W)
Y_rtn = X @ W_rtn
mse_rtn = np.mean((Y_true - Y_rtn) ** 2)

# Step 3: Implement AWQ (Activation-Aware Channel Scaling)
# Protect channel 2 by scaling weights up before quantization and activations down!
s = np.mean(np.abs(X), axis=0) ** 0.5 # Per-channel scaling vector
s = s / np.mean(s) # Normalized

# Equivalent Transformation: Y = (X * s^-1) @ (s * W)
X_smooth = X / s
W_scaled = s[:, np.newaxis] * W

# Quantize the scaled weights
W_scaled_q, _ = quantize_int4(W_scaled)
# Output projection: X_smooth @ W_scaled_q
Y_awq = X_smooth @ W_scaled_q
mse_awq = np.mean((Y_true - Y_awq) ** 2)

print("\n--- Quantization Performance Comparison ---")
print(f"Naive RTN INT4 Reconstruction MSE: {mse_rtn:.4f}")
print(f"AWQ Protected INT4 Reconstruction MSE: {mse_awq:.4f}")
print(f"Error Reduction from AWQ Protection: {(mse_rtn - mse_awq) / mse_rtn * 100:.1f}%!")

### 3. Step-by-Step Code Walkthrough
1. **Outlier Problem:** In RTN quantization, the error in Channel 2 gets multiplied by an activation magnitude of 80+, destroying output accuracy.
2. **Channel Re-balancing:** AWQ scales weights up in salient channels, reducing quantization grid truncation error.
3. **Exact Mathematical Invariance:** $X s^{-1} \cdot s W = X W$. The transformation is mathematically lossless in FP16, dramatically reducing error in INT4.

---

### 4. Industry Applications & Trade-offs
- **AutoAWQ / vLLM:** Used to serve 4-bit quantized versions of LLaMA-3 and Mistral with virtually zero loss in MMLU benchmark scores.
- **Speedup:** Reduces memory bandwidth requirements by 70%, yielding 3x faster generation on memory-bound GPUs.

---

### 5. Top Interview Questions & Answers
**Q1: Why are activations harder to quantize than weights?**
> **Answer:** Weights are static and known ahead of time, allowing offline optimization of scales. Activations depend on dynamic user input and exhibit systematic outlier channels with magnitudes 100x larger than normal tokens. Quantizing activations to 8 bits causes severe precision loss unless outlier channels are isolated or smoothed (SmoothQuant).

**Q2: What is the difference between Weight-Only Quantization (W4A16) and Weight-Activation Quantization (W8A8)?**
> **Answer:** W4A16 quantizes weights to 4 bits but dequantizes to FP16 before matrix multiplication in SRAM; it accelerates memory-bound generation for small batch sizes. W8A8 quantizes both weights and activations to INT8, utilizing high-throughput INT8 Tensor Cores for large batch sizes.